## Group Membership Ingestion

Fetches group membership data from the Databricks workspace SCIM API and writes it to Unity Catalog.

**Target table:** `catalog_40_copper_uc_metadata.metadata.group_membership`

> **Note:** This notebook requires **Workspace Admin** privileges to list group memberships.
> Currently runs under the interactive user's credentials. See README for the TODO to productionise with a dedicated service principal.

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

TARGET_TABLE = "catalog_40_copper_uc_metadata.metadata.group_membership"

print(f"Target table: {TARGET_TABLE}")
print(f"Authenticated as: {w.current_user.me().user_name}")

In [0]:
groups = list(w.groups.list(attributes="id,displayName,members"))
print(f"Groups found: {len(groups)}")

# Build user ID → email lookup
users = list(w.users.list(attributes="id,userName"))
user_email_lookup = {u.id: u.user_name for u in users if u.user_name}
print(f"Users found: {len(user_email_lookup)}")

# Flatten into membership records
ref_type_map = {
    "Users": "USER",
    "Groups": "GROUP",
    "ServicePrincipals": "SERVICE_PRINCIPAL",
}

memberships = []

for group in groups:
    if not group.members:
        continue
    for member in group.members:
        member_type = "UNKNOWN"
        if member.ref:
            ref_prefix = member.ref.split("/")[0]
            member_type = ref_type_map.get(ref_prefix, ref_prefix)

        memberships.append({
            "group_id": group.id,
            "group_name": group.display_name,
            "member_id": member.value,
            "member_display": member.display,
            "member_type": member_type,
            "member_email": user_email_lookup.get(member.value),
        })

print(f"Total membership records: {len(memberships)}")

In [0]:
from pyspark.sql.types import StructType, StructField, StringType

schema = StructType([
    StructField("group_id", StringType()),
    StructField("group_name", StringType()),
    StructField("member_id", StringType()),
    StructField("member_display", StringType()),
    StructField("member_type", StringType()),
    StructField("member_email", StringType()),
])

df = spark.createDataFrame(memberships, schema=schema)

print(f"DataFrame rows: {df.count()}")
display(df.limit(10))

In [0]:
df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TARGET_TABLE)

row_count = spark.table(TARGET_TABLE).count()
print(f"Written to {TARGET_TABLE} — {row_count} rows")

In [0]:
%sql
SELECT
  member_type,
  count(DISTINCT group_name) AS groups,
  count(DISTINCT(member_email)) AS users,
  count(*) AS memberships
FROM catalog_40_copper_uc_metadata.metadata.group_membership
GROUP BY member_type
ORDER BY memberships DESC